In [ ]:
def main(datasources, start_date, end_date):
    """
    BigAlpha 2026：自适应微观结构反转因子

    核心逻辑：
    1. 收盘价相对当日 VWAP 过度偏离 -> 短期价格修复
    2. 根据实体长度 / 日内振幅判断趋势还是噪声
    3. 用盘口买卖不平衡识别价格吸收现象
    4. 使用过去 3 个交易日平滑，降低单日噪声

    输出：
        date, instrument, factor
    """

    import numpy as np
    import pandas as pd
    import dai

    # ---------------------------------------------------------
    # 1. 评估系统会替换这个表名，绝对不能硬编码训练集表名
    # ---------------------------------------------------------
    bar1m = datasources["bar1m"]

    start_ts = pd.to_datetime(start_date)
    end_ts = pd.to_datetime(end_date)

    # 3日平滑只需要少量历史，这里取20个自然日，覆盖周末和节假日
    query_start_ts = start_ts - pd.Timedelta(days=20)
    query_start = query_start_ts.strftime("%Y-%m-%d %H:%M:%S")

    # ---------------------------------------------------------
    # 2. 优先运行完整版：价格 + 成交 + 三档盘口
    # ---------------------------------------------------------
    sql_full = f"""
    WITH daily_data AS (
        SELECT
            date::DATE::DATETIME AS date,
            instrument,

            FIRST(close ORDER BY date) AS open_px,
            LAST(close ORDER BY date)  AS close_px,

            MAX(close) AS high_px,
            MIN(close) AS low_px,

            SUM(amount) / NULLIF(SUM(volume), 0) AS vwap_px,
            SUM(amount) AS turnover,

            AVG(
                (
                    COALESCE(bid_volume1, 0)
                    + COALESCE(bid_volume2, 0)
                    + COALESCE(bid_volume3, 0)
                    - COALESCE(ask_volume1, 0)
                    - COALESCE(ask_volume2, 0)
                    - COALESCE(ask_volume3, 0)
                ) * 1.0
                /
                NULLIF(
                    COALESCE(bid_volume1, 0)
                    + COALESCE(bid_volume2, 0)
                    + COALESCE(bid_volume3, 0)
                    + COALESCE(ask_volume1, 0)
                    + COALESCE(ask_volume2, 0)
                    + COALESCE(ask_volume3, 0),
                    0
                )
            ) AS book_imbalance

        FROM {bar1m}

        WHERE
            close > 0
            AND volume >= 0

        GROUP BY
            date::DATE,
            instrument
    )

    SELECT
        date,
        instrument,
        open_px,
        close_px,
        high_px,
        low_px,
        vwap_px,
        turnover,
        book_imbalance

    FROM daily_data

    ORDER BY
        instrument,
        date
    """

    # ---------------------------------------------------------
    # 3. 如果盘口字段在某个环境出现异常，自动退回基础版本
    # ---------------------------------------------------------
    sql_basic = f"""
    WITH daily_data AS (
        SELECT
            date::DATE::DATETIME AS date,
            instrument,

            FIRST(close ORDER BY date) AS open_px,
            LAST(close ORDER BY date)  AS close_px,

            MAX(close) AS high_px,
            MIN(close) AS low_px,

            SUM(amount) / NULLIF(SUM(volume), 0) AS vwap_px,
            SUM(amount) AS turnover

        FROM {bar1m}

        WHERE
            close > 0
            AND volume >= 0

        GROUP BY
            date::DATE,
            instrument
    )

    SELECT
        date,
        instrument,
        open_px,
        close_px,
        high_px,
        low_px,
        vwap_px,
        turnover,
        0.0 AS book_imbalance

    FROM daily_data

    ORDER BY
        instrument,
        date
    """

    try:
        daily = dai.query(
            sql_full,
            filters={
                "date": [
                    query_start,
                    end_ts.strftime("%Y-%m-%d %H:%M:%S"),
                ]
            },
            compression=True,
        ).df()

    except Exception:
        daily = dai.query(
            sql_basic,
            filters={
                "date": [
                    query_start,
                    end_ts.strftime("%Y-%m-%d %H:%M:%S"),
                ]
            },
            compression=True,
        ).df()

    # ---------------------------------------------------------
    # 4. 读取正式股票池
    # ---------------------------------------------------------
    pool = dai.query(
        """
        SELECT
            date,
            instrument
        FROM bigalpha_2026_instruments
        """,
        filters={
            "date": [
                start_ts.strftime("%Y-%m-%d %H:%M:%S"),
                end_ts.strftime("%Y-%m-%d %H:%M:%S"),
            ]
        },
        compression=True,
    ).df()

    pool["date"] = pd.to_datetime(pool["date"])

    # 极端情况下没有行情，也返回合法的中性因子
    if daily.empty:
        pool = pool.drop_duplicates(["date", "instrument"])
        pool["factor"] = 0.0
        return (
            pool[["date", "instrument", "factor"]]
            .sort_values(["date", "instrument"])
            .reset_index(drop=True)
        )

    # ---------------------------------------------------------
    # 5. 数据清洗
    # ---------------------------------------------------------
    daily["date"] = pd.to_datetime(daily["date"])

    numeric_cols = [
        "open_px",
        "close_px",
        "high_px",
        "low_px",
        "vwap_px",
        "turnover",
        "book_imbalance",
    ]

    for col in numeric_cols:
        daily[col] = pd.to_numeric(daily[col], errors="coerce")

    daily = daily.drop_duplicates(
        subset=["date", "instrument"],
        keep="last",
    )

    eps = 1e-12

    # VWAP偶尔缺失时，退回当天收盘价
    daily["vwap_px"] = daily["vwap_px"].where(
        daily["vwap_px"] > 0,
        daily["close_px"],
    )

    # ---------------------------------------------------------
    # 6. 构建基础变量
    # ---------------------------------------------------------

    # 当日首个成交价格到收盘价格的收益
    daily["day_ret"] = (
        daily["close_px"]
        / daily["open_px"].clip(lower=eps)
        - 1.0
    )

    # 收盘价格偏离全天成交均价的程度
    daily["vwap_dev"] = (
        daily["close_px"]
        / daily["vwap_px"].clip(lower=eps)
        - 1.0
    )

    # 当天价格振幅
    daily["price_range"] = (
        daily["high_px"] - daily["low_px"]
    ).clip(lower=eps)

    # 实体长度 / 全天振幅：
    # 越接近1，说明价格方向运动较连贯；
    # 越接近0，说明当天更多是来回震荡。
    daily["body_efficiency"] = (
        (daily["close_px"] - daily["open_px"]).abs()
        / daily["price_range"]
    ).clip(lower=0.0, upper=1.0)

    # 收盘价处于全天价格区间的位置
    daily["close_position"] = (
        (daily["close_px"] - daily["low_px"])
        / daily["price_range"]
        - 0.5
    ).clip(lower=-0.5, upper=0.5)

    daily["book_imbalance"] = (
        daily["book_imbalance"]
        .replace([np.inf, -np.inf], np.nan)
        .fillna(0.0)
        .clip(lower=-1.0, upper=1.0)
    )

    # ---------------------------------------------------------
    # 7. 每日横截面排名
    #
    # 这里直接按照“列名”分组排名，
    # 避免之前把Series误传给辅助函数的问题。
    # ---------------------------------------------------------
    def centered_rank(column_name):
        ranks = daily.groupby(
            "date",
            sort=False,
        )[column_name].rank(
            method="average",
            pct=True,
        )

        return ranks.fillna(0.5) - 0.5

    daily["rank_ret"] = centered_rank("day_ret")
    daily["rank_vwap"] = centered_rank("vwap_dev")
    daily["rank_book"] = centered_rank("book_imbalance")
    daily["rank_close_position"] = centered_rank("close_position")

    # ---------------------------------------------------------
    # 8. 组合因子
    # ---------------------------------------------------------

    # 当实体较长时，更偏向趋势延续；
    # 当实体较短、震荡较强时，更偏向价格反转。
    daily["regime"] = (
        2.0 * daily["body_efficiency"] - 0.60
    )

    # 价格吸收：
    # 盘口买盘较强但当天价格偏弱，可能意味着卖压被吸收；
    # 盘口卖盘较强但价格偏强，可能意味着上涨阻力。
    daily["absorption"] = (
        daily["rank_book"] - daily["rank_ret"]
    )

    daily["raw_factor"] = (
        # 收盘显著高于VWAP时偏向反转，低于VWAP时偏向修复
        -0.42 * daily["rank_vwap"]

        # 根据价格运动效率，在趋势和反转之间自适应切换
        + 0.28 * daily["rank_ret"] * daily["regime"]

        # 盘口与价格方向不一致时的吸收信号
        + 0.20 * daily["absorption"]

        # 高效率且收于当日极端位置时，保留少量趋势信息
        + 0.10
        * daily["rank_close_position"]
        * daily["body_efficiency"]
    )

    daily["raw_factor"] = (
        daily["raw_factor"]
        .replace([np.inf, -np.inf], np.nan)
        .fillna(0.0)
    )

    # ---------------------------------------------------------
    # 9. 使用过去3个交易日进行平滑
    #
    # rolling只使用当前及过去数据，不使用未来数据。
    # ---------------------------------------------------------
    daily = daily.sort_values(
        ["instrument", "date"]
    ).reset_index(drop=True)

    daily["signal"] = (
        daily.groupby(
            "instrument",
            sort=False,
        )["raw_factor"]
        .transform(
            lambda series: series.rolling(
                window=3,
                min_periods=1,
            ).mean()
        )
    )

    # 计算完历史平滑后，再裁回正式评估区间
    daily = daily[
        (daily["date"] >= start_ts)
        & (daily["date"] <= end_ts)
    ].copy()

    # ---------------------------------------------------------
    # 10. 以股票池为主体左连接
    #
    # 即使个别股票当天停牌或行情缺失，也不会出现覆盖度不足。
    # ---------------------------------------------------------
    output = pool.merge(
        daily[["date", "instrument", "signal"]],
        how="left",
        on=["date", "instrument"],
    )

    output["signal"] = (
        pd.to_numeric(output["signal"], errors="coerce")
        .replace([np.inf, -np.inf], np.nan)
    )

    # 先以每天的中位数填充，再以0兜底
    daily_median = output.groupby(
        "date",
        sort=False,
    )["signal"].transform("median")

    output["signal"] = (
        output["signal"]
        .fillna(daily_median)
        .fillna(0.0)
    )

    # ---------------------------------------------------------
    # 11. 最终横截面排名
    #
    # 统一到[-0.5, 0.5]附近，避免极端值影响。
    # ---------------------------------------------------------
    output["factor"] = (
        output.groupby(
            "date",
            sort=False,
        )["signal"]
        .rank(
            method="average",
            pct=True,
        )
        - 0.5
    )

    output["factor"] = (
        pd.to_numeric(output["factor"], errors="coerce")
        .replace([np.inf, -np.inf], 0.0)
        .fillna(0.0)
        .astype("float64")
    )

    # ---------------------------------------------------------
    # 12. 严格整理最终格式
    # ---------------------------------------------------------
    output = (
        output[["date", "instrument", "factor"]]
        .drop_duplicates(
            subset=["date", "instrument"],
            keep="last",
        )
        .sort_values(["date", "instrument"])
        .reset_index(drop=True)
    )

    return output